# Group Classification Inference (Unified Config Pipeline)

This notebook runs the unified `inference_pipeline` using a config-only setup.


## 1) Environment Setup


In [ ]:
from pathlib import Path

from pioneerml.common.integration.zenml import load_step_output
from pioneerml.common.integration.zenml import utils as zenml_utils
from pioneerml.pipelines.inference import inference_pipeline

PROJECT_ROOT = zenml_utils.find_project_root()
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)


## 2) Select Input Files and Output Location


In [ ]:
# Input parquet files
_data_dir = Path(PROJECT_ROOT) / "data"
parquet_paths = sorted(_data_dir.glob("ml_output_*.parquet"))

# Optional: use a subset for quick checks
parquet_paths = parquet_paths[:1]

parquet_paths = [str(p.resolve()) for p in parquet_paths]
if not parquet_paths:
    raise RuntimeError(f"No parquet files found in {_data_dir}")

# Model path override (None => auto-resolve latest in trained_models/<model_subdir>)
model_path = None

# Output location
output_dir = str((Path(PROJECT_ROOT) / "artifacts" / "inference_small" / "group_classifier").resolve())


## 3) Reusable Config Helpers


In [ ]:
def make_model_handle_builder_cfg(*, model_subdir: str, model_type: str = "script", model_path: str | None = None) -> dict:
    return {
        "model_handle": {
            "type": str(model_type),
            "config": {
                "model_subdir": str(model_subdir),
                "model_path": model_path,
            },
        }
    }


def make_inference_loader_manager_cfg(*, input_sources_spec: dict, batch_size: int = 64) -> dict:
    return {
        "type": "config",
        "config": {
            "input_sources_spec": dict(input_sources_spec),
            "input_backend_name": "parquet",
            "defaults": {
                "type": "group_classifier",
                "config": {
                    "mode": "inference",
                    "batch_size": int(batch_size),
                    "chunk_row_groups": 4,
                    "chunk_workers": 0,
                    "sample_fraction": 1.0,
                    "split_seed": 0,
                },
            },
            "loaders": {
                "inference_loader": {
                    "config": {
                        "mode": "inference",
                        "shuffle_batches": False,
                        "log_diagnostics": False,
                    },
                },
            },
        },
    }


def make_writer_cfg(*, output_dir: str, write_timestamped: bool = True) -> dict:
    return {
        "type": "group_classification",
        "config": {
            "output_backend_name": "parquet",
            "fallback_output_dir": "artifacts/inference_small/group_classifier",
            "output_dir": str(output_dir),
            "output_path": None,
            "streaming": True,
            "write_timestamped": bool(write_timestamped),
            "timestamp": None,
            "writer_params": {},
        },
    }


## 4) Build Step Config Blocks


In [ ]:
input_sources_spec = {
    "main_sources": parquet_paths,
    "optional_sources_by_name": {},
    "source_type": "file",
}

model_handle_builder_cfg = make_model_handle_builder_cfg(
    model_subdir="groupclassifier",
    model_type="script",
    model_path=model_path,
)

inference_cfg = {
    "runtime": {
        "prefer_cuda": True,
    },
    "writer": make_writer_cfg(output_dir=output_dir, write_timestamped=True),
    "loader_manager": make_inference_loader_manager_cfg(
        input_sources_spec=input_sources_spec,
        batch_size=64,
    ),
}


## 5) Assemble `pipeline_config` and Run


In [ ]:
pipeline_config = {
    "model_handle_builder": model_handle_builder_cfg,
    "inference": inference_cfg,
}

run = inference_pipeline.with_options(enable_cache=False)(
    pipeline_config=pipeline_config,
)


## 6) Inspect Inference Outputs


In [ ]:
inference_output = load_step_output(run, "run_inference")
print("inference_output:", inference_output)

predictions_paths = [Path(p) for p in (inference_output.get("predictions_paths") or [])]
if not predictions_paths and inference_output.get("predictions_path"):
    predictions_paths = [Path(inference_output["predictions_path"])]

print("predictions_paths:")
for p in predictions_paths:
    print(" ", p)


## 7) Optional: Quick Parquet Validation


In [ ]:
import gc

import pyarrow as pa
import pyarrow.parquet as pq

if not predictions_paths:
    raise RuntimeError("No prediction parquet files were exported.")

pf = pq.ParquetFile(predictions_paths[0])
print("file:", predictions_paths[0])
print("rows:", pf.metadata.num_rows)
print(pf.schema_arrow)

if pf.num_row_groups > 0:
    sample = pf.read_row_group(0).slice(0, 3)
    print(sample)
else:
    sample = None
    print("No row groups found.")

del sample, pf
gc.collect()
pa.default_memory_pool().release_unused()
